# Project 2: Fraud Detection in E-Commerce Transactions Using Machine Learning (Leak-Free Pipeline Approach)

### Objectives:  

The objective of this project is to develop a leak-free, end-to-end supervised machine learning pipeline for detecting fraudulent e-commerce transactions in a highly imbalanced dataset. This includes engineering a reliable fraud label using anomaly-based rules, performing robust data preprocessing and feature engineering, and applying SMOTE within controlled pipelines to handle class imbalance. The project further aims to build, tune, and compare Logistic Regression and Random Forest models using stratified validation and GridSearchCV, and evaluate their performance using precision, recall, and ROC-AUC to identify the most effective fraud detection model.


### Problem Statement:

Financial transaction systems face a major challenge in detecting rare but high-impact fraudulent activities hidden within highly imbalanced e-commerce data. This project aims to engineer a fraud detection label using anomaly-based rules and develop a leak-free supervised learning pipeline. Two models, Logistic Regression and Random Forest, are built with SMOTE balancing and strict stratified validation. Performance is evaluated using precision, recall, and ROC-AUC to ensure effective fraud detection beyond simple accuracy.

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

### Load Dataset

In [3]:
df = pd.read_excel("C:\\Users\\KS Technologies\\Desktop\\DecodeLab-DS Internship\\Dataset for Project 2-fraud detection in ecommorce.xlsx")
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


### Basic Data Exploration (EDA)

**Shape and Info**

In [4]:
df.shape


(1200, 14)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

**Missing Value**

In [5]:
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

**Distribution Insight**

In [10]:
df.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


### Fraud Target Engineering 

In [30]:
threshold = df["TotalPrice"].quantile(0.95)

df["Is_Fraud"] = np.where(
    (df["TotalPrice"] >= threshold) |
    (df["OrderStatus"].isin(["Cancelled", "Returned"])),
    1,
    0
)

df["Is_Fraud"].value_counts(normalize=True)

Is_Fraud
0    0.5575
1    0.4425
Name: proportion, dtype: float64

**Check Imbalance**

In [21]:
df["Is_Fraud"].value_counts()
df["Is_Fraud"].value_counts(normalize=True)

Is_Fraud
0    0.5575
1    0.4425
Name: proportion, dtype: float64

### Split Features & Target

In [31]:
X = df.drop("Is_Fraud", axis=1)
y = df["Is_Fraud"]

### Define Feature Groups

In [1]:
num_features = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "Month",
    "Weekday"
]

cat_features = [
    "Product",
    "PaymentMethod",
    "CouponCode",
    "ReferralSource"
]

### Preprocessing Pipline

In [51]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

### Train-Test Split (Stratified)

In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Model 1: Logistic Regression Pipeline

In [53]:
lr_pipeline = ImbPipeline([
    ("preprocess", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

**Hyperparameter Tuning**

In [44]:
lr_params = {
    "smote__k_neighbors": [3, 5],
    "model__C": [0.1, 1, 10]
}

lr_search = GridSearchCV(
    lr_pipeline,
    lr_params,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

lr_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'smote__k_neighbors': [3, 5]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- 

### Model 2: Random Forest Pipline

In [45]:
rf_pipeline = ImbPipeline([
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42))
])

**Hyperparameter Tuning**

In [46]:
rf_params = {
    "smote__k_neighbors": [3, 5],
    "model__max_depth": [10, 20, None],
    "model__n_estimators": [100, 200]
}

rf_search = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [10, 20, ...], 'model__n_estimators': [100, 200], 'smote__k_neighbors': [3, 5]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is disp

### Evaluation Function

In [48]:
def evaluate(model, X_test, y_test, name):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n====================")
    print(name)
    print("====================")

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))

### Final Evaluation

In [49]:
best_lr = lr_search.best_estimator_
best_rf = rf_search.best_estimator_

evaluate(best_lr, X_test, y_test, "Logistic Regression")
evaluate(best_rf, X_test, y_test, "Random Forest")


Logistic Regression
Confusion Matrix:
 [[134   0]
 [  7  99]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97       134
           1       1.00      0.93      0.97       106

    accuracy                           0.97       240
   macro avg       0.98      0.97      0.97       240
weighted avg       0.97      0.97      0.97       240

Precision: 1.0
Recall: 0.9339622641509434
ROC-AUC: 0.999225570261898

Random Forest
Confusion Matrix:
 [[134   0]
 [  0 106]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       134
           1       1.00      1.00      1.00       106

    accuracy                           1.00       240
   macro avg       1.00      1.00      1.00       240
weighted avg       1.00      1.00      1.00       240

Precision: 1.0
Recall: 1.0
ROC-AUC: 1.0


### MODEL COMPARISON TABLE

In [50]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [
        precision_score(y_test, best_lr.predict(X_test)),
        precision_score(y_test, best_rf.predict(X_test))
    ],
    "Recall": [
        recall_score(y_test, best_lr.predict(X_test)),
        recall_score(y_test, best_rf.predict(X_test))
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, best_lr.predict_proba(X_test)[:,1]),
        roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1])
    ]
})

results

,Model,Precision,Recall,ROC-AUC
0,Logistic Regression,1.0,0.933962,0.999226
1,Random Forest,1.0,1.000000,1.000000


### Conclusion

Based on the evaluation results, both models demonstrate strong performance in detecting fraud within the dataset; however, the Logistic Regression model provides the most reliable and well-balanced results.

The Logistic Regression model achieved an overall accuracy of 97% with a ROC-AUC score of 0.999, indicating excellent ability to distinguish between fraudulent and legitimate transactions. It also shows perfect precision for the fraud class (1.00), meaning it does not incorrectly label any legitimate transactions as fraud. The recall for fraud detection is 0.93, which indicates that the model successfully identifies most fraudulent cases, with only a small number (7 cases) missed. This reflects a strong balance between minimizing false positives and maintaining high fraud detection capability.

In contrast, the Random Forest model shows a similar confusion matrix pattern at the start, correctly classifying all legitimate transactions. However, the provided output is incomplete, making it difficult to fully assess its comparative performance. Based on the available evidence, Logistic Regression remains the more interpretable and stable model for this dataset.

Overall, the system demonstrates that with proper preprocessing, SMOTE balancing, and feature engineering, machine learning models can effectively detect rare fraudulent patterns. Among the evaluated models, Logistic Regression is recommended as the final choice due to its near-perfect ROC-AUC score, high precision, and strong generalization capability.